In [4]:
import pandas as pd
from pathlib import Path

# 1. Define paths (relative to notebooks/ folder)
raw_path = Path("../data/raw/transfermarkt")
processed_path = Path("../data/processed")

# Ensure processed directory exists
processed_path.mkdir(parents=True, exist_ok=True)

print("⏳ Loading Transfermarkt source files...")
appearances = pd.read_csv(raw_path / "appearances.csv")
players = pd.read_csv(raw_path / "players.csv")
games = pd.read_csv(raw_path / "games.csv")
print("✅ Source files loaded.\n")

# 2. Perform the left-join pipeline exactly as requested
print("🔗 Merging appearances, players, and games datasets...")
player_match_stats = (
    appearances
    .merge(
        players[
            [
                'player_id',
                'name',
                'position',
                'sub_position',
                'country_of_citizenship',
                'market_value_in_eur'
            ]
        ],
        on='player_id',
        how='left'
    )
    .merge(
        games[
            [
                'game_id',
                'season',
                'competition_id',
                'competition_type',
                'home_club_name',
                'away_club_name'
            ]
        ],
        on='game_id',
        how='left'
    )
)
print("✅ Merge operation complete.\n")

# ==========================================
# IMMEDIATE INSPECTIONS
# ==========================================
print("=" * 80)
print("📊 INSPECTION 1: SHAPE")
print("=" * 80)
print(f"Shape: {player_match_stats.shape}\n")

print("=" * 80)
print("👀 INSPECTION 2: HEAD")
print("=" * 80)
display(player_match_stats.head())
print("\n")

print("=" * 80)
print("💡 INSPECTION 3: INFO")
print("=" * 80)
player_match_stats.info()
print("\n")

print("=" * 80)
print("❓ INSPECTION 4: SORTED MISSING VALUE COUNTS")
print("=" * 80)
missing_counts = player_match_stats.isnull().sum().sort_values(ascending=False)
print(missing_counts)
print("\n")

# ==========================================
# FILE EXPORT (PARQUET PREFERRED)
# ==========================================
output_file = processed_path / "player_match_stats.parquet"
print(f"💾 Saving merged dataset to: {output_file}...")

try:
    player_match_stats.to_parquet(output_file, index=False)
    print("🎉 File saved successfully in Parquet format!")
except ImportError:
    print("⚠️  'pyarrow' or 'fastparquet' missing in your environment.")
    print("⚙️ Running fallback saving strategy to CSV...")
    fallback_file = processed_path / "player_match_stats.csv"
    player_match_stats.to_csv(fallback_file, index=False)
    print(f"🎉 File saved successfully in CSV format instead: {fallback_file}")


⏳ Loading Transfermarkt source files...
✅ Source files loaded.

🔗 Merging appearances, players, and games datasets...
✅ Merge operation complete.

📊 INSPECTION 1: SHAPE
Shape: (1885688, 23)

👀 INSPECTION 2: HEAD


,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id_x,yellow_cards,red_cards,...,name,position,sub_position,country_of_citizenship,market_value_in_eur,season,competition_id_y,competition_type,home_club_name,away_club_name
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,...,Aurélien Joachim,Attack,Centre-Forward,Luxembourg,75000.0,2012,CLQ,international_cup,F91 Dudelange,SP Tre Penne
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,...,Ruslan Abyshov,Defender,Centre-Back,Azerbaijan,25000.0,2012,ELQ,other,Xäzär Länkäran,Kalju FC
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,...,Sander Puri,Midfield,Central Midfield,Estonia,100000.0,2012,ELQ,other,Kuopion Palloseura,Llanelli AFC
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,...,Vegar Hedenstad,Defender,Right-Back,Norway,350000.0,2012,ELQ,other,JJK Jyväskylä,Stabæk Fotball
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,...,Markus Henriksen,Defender,Centre-Back,Norway,800000.0,2012,ELQ,other,Crusaders FC,Rosenborg Ballklub




💡 INSPECTION 3: INFO
<class 'pandas.DataFrame'>
RangeIndex: 1885688 entries, 0 to 1885687
Data columns (total 23 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   appearance_id           str    
 1   game_id                 int64  
 2   player_id               int64  
 3   player_club_id          int64  
 4   player_current_club_id  int64  
 5   date                    str    
 6   player_name             str    
 7   competition_id_x        str    
 8   yellow_cards            int64  
 9   red_cards               int64  
 10  goals                   int64  
 11  assists                 int64  
 12  minutes_played          int64  
 13  name                    str    
 14  position                str    
 15  sub_position            str    
 16  country_of_citizenship  str    
 17  market_value_in_eur     float64
 18  season                  int64  
 19  competition_id_y        str    
 20  competition_type        str    
 21  home_club_name     

In [19]:


print("⏳ Step 0: Merging Date of Birth from players file...")
# Load players to grab date_of_birth (since it wasn't in your previous join)
players_dob = pd.read_csv(raw_path / "players.csv", usecols=['player_id', 'date_of_birth'])

# Initialize your working copy
player_match_stats_clean = player_match_stats.copy()

# Add date_of_birth to your working dataframe
player_match_stats_clean = player_match_stats_clean.merge(players_dob, on='player_id', how='left')

# ==========================================
# 1. REMOVE DUPLICATE COMPETITION COLUMN
# ==========================================
# Run the check and print the result explicitly
comp_columns_match = (
    player_match_stats_clean['competition_id_x'] 
    == 
    player_match_stats_clean['competition_id_y']
).all()

if comp_columns_match:
    player_match_stats_clean = (
        player_match_stats_clean
        .drop(columns=['competition_id_y'])
        .rename(columns={'competition_id_x': 'competition_id'})
    )

# ==========================================
# 2 & 3. CONVERT DATES & PREPARE AGE FEATURE
# ==========================================
# Convert match date
player_match_stats_clean['date'] = pd.to_datetime(player_match_stats_clean['date'])

# Convert Date of Birth safely
player_match_stats_clean['date_of_birth'] = pd.to_datetime(
    player_match_stats_clean['date_of_birth'], 
    errors='coerce'
)

# Optional: Calculate age at the exact time of the match
# player_match_stats_clean['age_at_match'] = (player_match_stats_clean['date'] - player_match_stats_clean['date_of_birth']).dt.days / 365.25

# ==========================================
# 4. SAVE CLEANED VERSION
# ==========================================
output_csv = processed_path / "player_match_stats_clean.csv"
player_match_stats_clean.to_csv(output_csv, index=False)


# ==========================================
# 5. ALL REQUESTED OUTPUTS (PRINTED TOGETHER)
# ==========================================
print("\n" + "="*60)
print("📊 FINAL VERIFICATION RESULTS PACK")
print("="*60)

# Check 1: Competition column equality
print(f"1. Competition Columns Match Exactly? -> {comp_columns_match}")

# Check 2: Total exact rows duplicated
total_dups = player_match_stats_clean.duplicated().sum()
print(f"2. Total Row Duplicates Across Entire Table -> {total_dups}")

# Check 3: Unique constraints check (Player + Match composite key)
key_dups = player_match_stats_clean.duplicated(subset=['player_id', 'game_id']).sum()
print(f"3. Duplicates on ['player_id', 'game_id'] Combination -> {key_dups}")
print("="*60)


⏳ Step 0: Merging Date of Birth from players file...

📊 FINAL VERIFICATION RESULTS PACK
1. Competition Columns Match Exactly? -> True
2. Total Row Duplicates Across Entire Table -> 0
3. Duplicates on ['player_id', 'game_id'] Combination -> 0
